In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping



In [25]:
data = pd.read_csv("/content/training_data_lowercase.csv",sep='\t', names=['label', 'title'])
print(data.shape)
data.fillna("",inplace=True)
print(data.head())


(34152, 2)
   label                                              title
0      0  donald trump sends out embarrassing new year‚s...
1      0  drunk bragging trump staffer started russian c...
2      0  sheriff david clarke becomes an internet joke ...
3      0  trump is so obsessed he even has obama‚s name ...
4      0  pope francis just called out donald trump duri...


In [26]:
data["text"] = data["title"].astype(str)

TRAIN / VALIDATION SPLIT

In [27]:
X_train, X_val, y_train, y_val = train_test_split(
    data["text"],
    data["label"],
    test_size=0.2,
    random_state=42
)


print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_val.shape}")
print(f"Training output size: {y_train.shape}")
print(f"Testing output size: {y_val.shape}")

Training set size: (27321,)
Testing set size: (6831,)
Training output size: (27321,)
Testing output size: (6831,)


TOKENIZATION

In [28]:
num_words = 10000
maxlen = 100

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

X_train_pad = pad_sequences(X_train_seq, maxlen=maxlen, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=maxlen, padding="post", truncating="post")


In [29]:
#Check vocabulary (word → number)
print("\nVocabulary:")
print(tokenizer.word_index)
#Check top words
print("\nTop words:")
for word, index in list(tokenizer.word_index.items())[:20]:
    print(word, "→", index)
#Check one sentence
print("\nOriginal text:")
print(X_train.iloc[0])
print("\nTokenized:")
print(X_train_seq[0])
#Check padded version
print("\nPadded:")
print(X_train_pad[0])
#Convert back
reverse_word_index = {v: k for k, v in tokenizer.word_index.items()}
decoded = [reverse_word_index.get(i, "?") for i in X_train_seq[0]]
print("\nDecoded:")
print(decoded)
print("\nVocabulary size:")
print(len(tokenizer.word_index))
print("\nSequence:")
print(X_train_seq[0])
print("\nShape:")
print(X_train_pad.shape)


Vocabulary:
{'<OOV>': 1, 'to': 2, 'trump': 3, 'in': 4, 'of': 5, 'for': 6, 'video': 7, 'on': 8, 'the': 9, 'u': 10, 's': 11, 'a': 12, 'with': 13, 'says': 14, 'and': 15, 'is': 16, 'obama': 17, 'after': 18, 'house': 19, 'at': 20, 'as': 21, 'his': 22, 'about': 23, 'over': 24, 'from': 25, 'by': 26, 'clinton': 27, 'white': 28, 'new': 29, 'trump‚s': 30, 'hillary': 31, 'just': 32, 'not': 33, 'will': 34, 'president': 35, 'bill': 36, 'he': 37, 'republican': 38, 'be': 39, 'russia': 40, 'out': 41, 'this': 42, 'senate': 43, 'it': 44, 'who': 45, 'that': 46, 'court': 47, 'state': 48, 'up': 49, 'donald': 50, 'are': 51, 'tax': 52, 'has': 53, 'election': 54, 'against': 55, 'republicans': 56, '‚': 57, 'him': 58, 'gop': 59, 'calls': 60, 'her': 61, 'breaking': 62, 'news': 63, "trump's": 64, 'north': 65, 'was': 66, 'how': 67, 'campaign': 68, 'why': 69, 'vote': 70, 'media': 71, 'you': 72, 'no': 73, 'black': 74, 'watch': 75, 'more': 76, 'korea': 77, 'have': 78, 'anti': 79, 'down': 80, 'gets': 81, 'senator': 8

BUILD MODEL

In [30]:
model = Sequential([
    Embedding(input_dim=num_words, output_dim=128, input_length=maxlen),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

TRAIN MODEL

In [31]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/10
854/854 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 0.9320 - loss: 0.1656 - val_accuracy: 0.9668 - val_loss: 0.0829
Epoch 2/10
854/854 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.9830 - loss: 0.0509 - val_accuracy: 0.9669 - val_loss: 0.0872
Epoch 3/10
854/854 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.9911 - loss: 0.0266 - val_accuracy: 0.9655 - val_loss: 0.0977


EVALUATE

In [32]:
val_probs = model.predict(X_val_pad)
val_preds = (val_probs > 0.5).astype("int32").flatten()

acc = accuracy_score(y_val, val_preds)
print("\nEmbedding Model Accuracy:", acc)

print("\nClassification Report:")
print(classification_report(y_val, val_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_preds))

214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

Embedding Model Accuracy: 0.9667691406821841

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      3529
           1       0.96      0.97      0.97      3302

    accuracy                           0.97      6831
   macro avg       0.97      0.97      0.97      6831
weighted avg       0.97      0.97      0.97      6831


Confusion Matrix:
[[3400  129]
 [  98 3204]]
